In [0]:
dbutils.widgets.text("landing_timestamp", "", "")
dbutils.widgets.text("table_name", "", "")
dbutils.widgets.text("merge_keys", "", "")
dbutils.widgets.text("load_type", "", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("="*60)
print("STEP 1: Widget Parameters")
print("="*60)
print(f"  landing_timestamp : {landing_timestamp}")
print(f"  table_name        : {table_name}")
print(f"  load_type         : {load_type}")
print(f"  merge_keys        : {merge_keys}")
print("="*60)

In [0]:
print("="*60)
print("STEP 2: Reading from Bronze")
print("="*60)

source_table = f"adb_dev_healthcare.bronze.{table_name}"
filter_expr = f"landing_timestamp = '{landing_timestamp}'"

print(f"  Source table  : {source_table}")
print(f"  Filter        : {filter_expr}")

bronze_df = spark.read.table(source_table).filter(filter_expr)
record_count = bronze_df.count()

print(f"  Records read  : {record_count}")
print(f"  Columns       : {bronze_df.columns}")
print("="*60)

display(bronze_df)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

print("="*60)
print("STEP 3: Adding Audit Columns")
print("="*60)
print("  Adding column : insert_timestamp = current_timestamp()")
print("  Adding column : update_timestamp = current_timestamp()")

updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp())\
    .withColumn("update_timestamp", current_timestamp())

print(f"  Final columns : {updated_df.columns}")
print("="*60)


In [0]:
from delta.tables import DeltaTable

target_table = f"adb_dev_healthcare.silver.{table_name}"

print("="*60)
print("STEP 4: Writing to Silver")
print("="*60)
print(f"  Target table : {target_table}")
print(f"  Load type    : {load_type.upper()}")

if load_type.upper() == "FULL":
    print("\n  [FULL LOAD] Overwriting entire target table...")
    print(f"  Query: updated_df.write.mode('overwrite').saveAsTable('{target_table}')")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("  [FULL LOAD] Complete.")

elif load_type.upper() == "APPEND":
    print("\n  [APPEND] Appending records to target table...")
    print(f"  Query: updated_df.write.mode('append').saveAsTable('{target_table}')")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("  [APPEND] Complete.")

elif load_type.upper() == "MERGE":
    keys = [k.strip() for k in merge_keys.split(",")]
    merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in keys])

    print(f"\n  [MERGE] Merge keys       : {keys}")
    print(f"  [MERGE] Merge condition  : {merge_condition}")

    table_exists = spark.catalog.tableExists(target_table)
    print(f"  [MERGE] Target exists    : {table_exists}")

    if table_exists:
        target_delta = DeltaTable.forName(spark, target_table)

        # Preserve original insert_timestamp for existing records; only update the rest
        update_set = {c: f"source.{c}" for c in updated_df.columns if c != "insert_timestamp"}

        print(f"\n  [MERGE] Update set (excludes insert_timestamp):")
        for col_name, col_expr in update_set.items():
            print(f"           {col_name} = {col_expr}")

        print(f"\n  [MERGE] Executing:")
        print(f"    MERGE INTO {target_table} AS target")
        print(f"    USING source_df AS source")
        print(f"    ON {merge_condition}")
        print(f"    WHEN MATCHED THEN UPDATE SET {{...{len(update_set)} columns...}}")
        print(f"    WHEN NOT MATCHED THEN INSERT *")

        target_delta.alias("target").merge(
            updated_df.alias("source"),
            merge_condition
        ).whenMatchedUpdate(
            set=update_set
        ).whenNotMatchedInsertAll(
        ).execute()
        print("  [MERGE] Complete.")
    else:
        print(f"  [MERGE] Target table does not exist. Creating with initial load...")
        print(f"  Query: updated_df.write.mode('overwrite').saveAsTable('{target_table}')")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("  [MERGE] Initial table created.")
else:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Expected 'FULL', 'APPEND', or 'MERGE'.")

print("="*60)

In [0]:
records_count = updated_df.count()

print("="*60)
print("STEP 5: Notebook Exit")
print("="*60)
print(f"  Total records processed : {records_count}")
print(f"  Exiting notebook with   : {records_count}")
print("="*60)

dbutils.notebook.exit(str(records_count))

In [0]:
print("tejas")